In [ ]:
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 pyyaml

In [ ]:
%%writefile pretokenize.py
from pathlib import Path
import os

import yaml
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_dataset


MODEL_CONFIG_NAME = "tiny-aya-base"  # <-- SET BEFORE RUNNING
CONFIGS_DIR = os.environ.get("CONFIGS_DIR")
# Set the condition to train. Must match a config in
# legesher/language-decoded-data.
#
# Available conditions:
#   "condition-1-en-32k"   - English code (full 32K)
#   "condition-1-en-5k"    - English code (5K subset)
#   "condition-2-zh-5k"    - Chinese keyword-swapped (5K)
#   "condition-2-es-5k"    - Spanish keyword-swapped (5K)
#   "condition-2-ur-5k"    - Urdu keyword-swapped (5K)
#   "condition-3-zh-5k"    - Chinese mixed native (5K)
CONDITION_NAME = ""  # <-- SET BEFORE RUNNING
if not MODEL_CONFIG_NAME:
    raise ValueError("MODEL_CONFIG_NAME must be set before running.")
if not CONDITION_NAME:
    raise ValueError("CONDITION_NAME must be set before running. See options above.")


def get_config_search_dirs():
    search_dirs = []
    if CONFIGS_DIR:
        search_dirs.append(Path(CONFIGS_DIR).expanduser())

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        search_dirs.append(base / "configs")
        search_dirs.append(base / "training" / "configs")

    unique_dirs = []
    seen = set()
    for config_dir in search_dirs:
        resolved_dir = config_dir.resolve(strict=False)
        if resolved_dir in seen:
            continue
        seen.add(resolved_dir)
        unique_dirs.append(resolved_dir)
    return unique_dirs


def validate_model_config(config, config_path):
    if not isinstance(config, dict):
        raise TypeError(f"Config at {config_path} must load to a dict.")

    required_keys = [
        "model_id",
        "max_seq_length",
        "target_modules",
        "lora_r",
        "lora_alpha",
        "lora_dropout",
        "learning_rate",
        "per_device_train_batch_size",
        "num_train_epochs",
        "warmup_ratio",
        "max_grad_norm",
        "use_unsloth",
    ]
    missing_keys = [key for key in required_keys if key not in config]
    if missing_keys:
        raise KeyError(f"Config at {config_path} is missing required keys: {missing_keys}")

    if not isinstance(config["model_id"], str):
        raise TypeError(f"Config key 'model_id' must be a string in {config_path}.")
    if not isinstance(config["max_seq_length"], int):
        raise TypeError(f"Config key 'max_seq_length' must be an int in {config_path}.")
    if not isinstance(config["target_modules"], list) or not all(isinstance(module, str) for module in config["target_modules"]):
        raise TypeError(f"Config key 'target_modules' must be a list of strings in {config_path}.")
    if not isinstance(config["use_unsloth"], bool):
        raise TypeError(f"Config key 'use_unsloth' must be a bool in {config_path}.")


def load_model_config(config_name):
    config_search_dirs = get_config_search_dirs()
    for config_dir in config_search_dirs:
        config_path = config_dir / f"{config_name}.yml"
        if config_path.exists():
            with config_path.open() as f:
                model_config = yaml.safe_load(f)
            validate_model_config(model_config, config_path)
            return model_config, config_path
    searched_paths = [str(config_dir / f"{config_name}.yml") for config_dir in config_search_dirs]
    raise FileNotFoundError(
        f"Could not find config '{config_name}'. Looked in: {searched_paths}. Set CONFIGS_DIR to override the search path."
    )


MODEL_CONFIG, MODEL_CONFIG_PATH = load_model_config(MODEL_CONFIG_NAME)
TRAIN_OUT = f"/kaggle/working/data/{MODEL_CONFIG_NAME}/{CONDITION_NAME}/train_tokenized"


def main():
    os.makedirs(os.path.dirname(TRAIN_OUT), exist_ok=True)

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("hf")

    if not MODEL_CONFIG.get("use_unsloth", True):
        raise ValueError("This notebook currently supports only use_unsloth=true configs.")

    print(f"Loaded model config from: {MODEL_CONFIG_PATH}")
    print(f"Model ID: {MODEL_CONFIG['model_id']}")

    # We only need the tokenizer here.
    _, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_CONFIG["model_id"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        load_in_4bit=MODEL_CONFIG.get("load_in_4bit", True),
        full_finetuning=MODEL_CONFIG.get("full_finetuning", False),
        token=hf_token,
    )

    train_dataset = load_dataset(
        "legesher/language-decoded-data",
        CONDITION_NAME,
        split="train",
    )

    def tokenize_fn(batch):
        return tokenizer(
            batch["code"],
            truncation=True,
            max_length=MODEL_CONFIG["max_seq_length"],
            padding=False,
        )

    # Keep only tokenized fields to reduce disk size.
    train_tokenized = train_dataset.map(
        tokenize_fn,
        batched=True,
        num_proc=8,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train dataset",
    )

    train_tokenized.save_to_disk(TRAIN_OUT)

    print(f"Saved train tokenized dataset to: {TRAIN_OUT}")
    print(train_tokenized)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
from pathlib import Path
import json
import os

import torch
import yaml
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_from_disk
from transformers import DataCollatorForLanguageModeling, set_seed
from trl import SFTTrainer, SFTConfig


MODEL_CONFIG_NAME = "tiny-aya-base"  # <-- SET BEFORE RUNNING
CONFIGS_DIR = os.environ.get("CONFIGS_DIR")
DEFAULT_TRAIN_SEEDS = [42, 123, 456, 789, 1024]
TRAIN_SEEDS_ENV = os.environ.get("TRAIN_SEEDS")
if TRAIN_SEEDS_ENV:
    TRAIN_SEEDS = [int(seed.strip()) for seed in TRAIN_SEEDS_ENV.split(",") if seed.strip()]
else:
    TRAIN_SEEDS = DEFAULT_TRAIN_SEEDS
TRAIN_SEED = int(os.environ.get("TRAIN_SEED", str(TRAIN_SEEDS[0])))
# Set the condition to train. Must match a config in
# legesher/language-decoded-data.
#
# Available conditions:
#   "condition-1-en-32k"   - English code (full 32K)
#   "condition-1-en-5k"    - English code (5K subset)
#   "condition-2-zh-5k"    - Chinese keyword-swapped (5K)
#   "condition-2-es-5k"    - Spanish keyword-swapped (5K)
#   "condition-2-ur-5k"    - Urdu keyword-swapped (5K)
#   "condition-3-zh-5k"    - Chinese mixed native (5K)
CONDITION_NAME = ""  # <-- SET BEFORE RUNNING
if not MODEL_CONFIG_NAME:
    raise ValueError("MODEL_CONFIG_NAME must be set before running.")
if not CONDITION_NAME:
    raise ValueError("CONDITION_NAME must be set before running. See options above.")


def get_config_search_dirs():
    search_dirs = []
    if CONFIGS_DIR:
        search_dirs.append(Path(CONFIGS_DIR).expanduser())

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        search_dirs.append(base / "configs")
        search_dirs.append(base / "training" / "configs")

    unique_dirs = []
    seen = set()
    for config_dir in search_dirs:
        resolved_dir = config_dir.resolve(strict=False)
        if resolved_dir in seen:
            continue
        seen.add(resolved_dir)
        unique_dirs.append(resolved_dir)
    return unique_dirs


def validate_model_config(config, config_path):
    if not isinstance(config, dict):
        raise TypeError(f"Config at {config_path} must load to a dict.")

    required_keys = [
        "model_id",
        "max_seq_length",
        "target_modules",
        "lora_r",
        "lora_alpha",
        "lora_dropout",
        "learning_rate",
        "per_device_train_batch_size",
        "num_train_epochs",
        "warmup_ratio",
        "max_grad_norm",
        "use_unsloth",
    ]
    missing_keys = [key for key in required_keys if key not in config]
    if missing_keys:
        raise KeyError(f"Config at {config_path} is missing required keys: {missing_keys}")

    if not isinstance(config["model_id"], str):
        raise TypeError(f"Config key 'model_id' must be a string in {config_path}.")
    if not isinstance(config["max_seq_length"], int):
        raise TypeError(f"Config key 'max_seq_length' must be an int in {config_path}.")
    if not isinstance(config["target_modules"], list) or not all(isinstance(module, str) for module in config["target_modules"]):
        raise TypeError(f"Config key 'target_modules' must be a list of strings in {config_path}.")
    if not isinstance(config["use_unsloth"], bool):
        raise TypeError(f"Config key 'use_unsloth' must be a bool in {config_path}.")


def load_model_config(config_name):
    config_search_dirs = get_config_search_dirs()
    for config_dir in config_search_dirs:
        config_path = config_dir / f"{config_name}.yml"
        if config_path.exists():
            with config_path.open() as f:
                model_config = yaml.safe_load(f)
            validate_model_config(model_config, config_path)
            return model_config, config_path
    searched_paths = [str(config_dir / f"{config_name}.yml") for config_dir in config_search_dirs]
    raise FileNotFoundError(
        f"Could not find config '{config_name}'. Looked in: {searched_paths}. Set CONFIGS_DIR to override the search path."
    )


def is_main_process():
    return int(os.environ.get("RANK", "0")) == 0


MODEL_CONFIG, MODEL_CONFIG_PATH = load_model_config(MODEL_CONFIG_NAME)
TRAIN_PATH = f"/kaggle/working/data/{MODEL_CONFIG_NAME}/{CONDITION_NAME}/train_tokenized"
OUTPUT_NAME = f"{CONDITION_NAME}-seed{TRAIN_SEED}"
OUTPUT_DIR = f"/kaggle/working/output/{MODEL_CONFIG_NAME}/{OUTPUT_NAME}"
UPLOAD_SUBFOLDER = f"{MODEL_CONFIG_NAME}/{OUTPUT_NAME}"


def main():
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("hf")

    if not MODEL_CONFIG.get("use_unsloth", True):
        raise ValueError("This notebook currently supports only use_unsloth=true configs.")

    # Helpful for near-OOM situations.
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    set_seed(TRAIN_SEED)

    if is_main_process():
        print(f"Loaded model config from: {MODEL_CONFIG_PATH}")
        print(f"Model ID: {MODEL_CONFIG['model_id']}")
        print(f"Condition: {CONDITION_NAME}")
        print(f"Training seed: {TRAIN_SEED}")

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_CONFIG["model_id"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        load_in_4bit=MODEL_CONFIG.get("load_in_4bit", True),
        full_finetuning=MODEL_CONFIG.get("full_finetuning", False),
        token=hf_token,
        # Do not set device_map="auto" under DDP
    )

    model = FastModel.get_peft_model(
        model,
        r=MODEL_CONFIG["lora_r"],
        lora_alpha=MODEL_CONFIG["lora_alpha"],
        lora_dropout=MODEL_CONFIG["lora_dropout"],
        bias=MODEL_CONFIG.get("bias", "none"),
        random_state=TRAIN_SEED,
        gradient_checkpointing=MODEL_CONFIG.get("gradient_checkpointing", "unsloth"),
        target_modules=MODEL_CONFIG["target_modules"],
    )

    train_dataset = load_from_disk(TRAIN_PATH)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    if is_main_process():
        gpu_stats = torch.cuda.get_device_properties(0)
        start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        max_memory = round(gpu_stats.total_memory / 1024**3, 3)
        print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
        print(f"{start_gpu_memory} GB of memory reserved.")
        print(train_dataset)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        data_collator=data_collator,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            max_seq_length=MODEL_CONFIG["max_seq_length"],
            max_grad_norm=MODEL_CONFIG["max_grad_norm"],
            per_device_train_batch_size=MODEL_CONFIG["per_device_train_batch_size"],
            gradient_accumulation_steps=MODEL_CONFIG.get("gradient_accumulation_steps", 1),
            warmup_ratio=MODEL_CONFIG["warmup_ratio"],
            num_train_epochs=MODEL_CONFIG["num_train_epochs"],
            learning_rate=MODEL_CONFIG["learning_rate"],
            packing=MODEL_CONFIG.get("packing", True),
            logging_steps=MODEL_CONFIG.get("logging_steps", 10),
            optim=MODEL_CONFIG.get("optim", "paged_adamw_8bit"),
            weight_decay=MODEL_CONFIG.get("weight_decay", 0.01),
            lr_scheduler_type=MODEL_CONFIG.get("lr_scheduler_type", "cosine"),
            report_to="none",
            fp16=MODEL_CONFIG.get("fp16", True),
            bf16=MODEL_CONFIG.get("bf16", False),
            eval_strategy=MODEL_CONFIG.get("eval_strategy", "no"),
            save_strategy=MODEL_CONFIG.get("save_strategy", "steps"),
            save_steps=MODEL_CONFIG.get("save_steps", 500),
            save_total_limit=MODEL_CONFIG.get("save_total_limit", 2),
            run_name=f"{MODEL_CONFIG_NAME}-{OUTPUT_NAME}",
            ddp_find_unused_parameters=False,
            seed=TRAIN_SEED,
            data_seed=TRAIN_SEED,
            dataloader_num_workers=MODEL_CONFIG.get("dataloader_num_workers", 2),
            dataloader_pin_memory=MODEL_CONFIG.get("dataloader_pin_memory", True),
            dataloader_persistent_workers=MODEL_CONFIG.get("dataloader_persistent_workers", True),
        ),
    )

    trainer_stats = trainer.train()

    if is_main_process():
        used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
        print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
        print(f"Peak reserved memory = {used_memory} GB.")

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        metrics_payload = {
            "model_config_name": MODEL_CONFIG_NAME,
            "condition_name": CONDITION_NAME,
            "seed": TRAIN_SEED,
            "output_name": OUTPUT_NAME,
            "train_result": trainer_stats.metrics,
            "log_history": trainer.state.log_history,
        }
        with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
            json.dump(metrics_payload, f, indent=2)

        print(f"Saved model + metrics to: {OUTPUT_DIR}")

        from huggingface_hub import HfApi

        api = HfApi()
        api.upload_folder(
            folder_path=OUTPUT_DIR,
            path_in_repo=UPLOAD_SUBFOLDER,
            repo_id="legesher/language-decoded-lora",
            repo_type="model",
            token=hf_token,
            # Only upload adapter weights + per-seed metrics, skip tokenizer files
            # (tokenizer is already in base model) and checkpoint folders.
            allow_patterns=["adapter_*", "training_metrics.json"],
        )
        print(f"Uploaded adapter + metrics to legesher/language-decoded-lora/{UPLOAD_SUBFOLDER}")


if __name__ == "__main__":
    main()


In [ ]:
!python pretokenize.py

In [ ]:
%%bash
DEFAULT_SEEDS=(42 123 456 789 1024)
NUM_TRAIN_SEEDS="${NUM_TRAIN_SEEDS:-5}"
TRAIN_SEEDS="${TRAIN_SEEDS:-}"

if [ -n "$TRAIN_SEEDS" ]; then
  IFS=',' read -r -a SEEDS <<< "$TRAIN_SEEDS"
else
  if [ "$NUM_TRAIN_SEEDS" -lt 1 ] || [ "$NUM_TRAIN_SEEDS" -gt "${#DEFAULT_SEEDS[@]}" ]; then
    echo "NUM_TRAIN_SEEDS must be between 1 and ${#DEFAULT_SEEDS[@]}" >&2
    exit 1
  fi
  SEEDS=("${DEFAULT_SEEDS[@]:0:$NUM_TRAIN_SEEDS}")
fi

for seed in "${SEEDS[@]}"; do
  TRAIN_SEED="$seed" torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py
done